In [1]:
# Check if a GPU is available
!nvidia-smi || echo 'No GPU available. You may be on a CPU-only runtime.'

Thu Jan  1 11:13:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

ln: failed to create symbolic link '/root/.ollama/models': No such file or directory
Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models


In [5]:
import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')

Ollama server started as background process (PID: 3094 )
Waited 5 seconds for the server to initialize.


In [6]:
# LLM
!ollama pull gemma3:4b

# SLM
!ollama pull gemma3:1b

In [7]:
!pip install -r '/content/drive/MyDrive/RAG/requirements.txt'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
%%time
import os
import math
import pandas as pd
import sys
from pathlib import Path

# Ensure the project root is in sys.path to allow imports from 'src'
project_root = '/content/drive/MyDrive/RAG'
if project_root not in sys.path:
    sys.path.append(project_root)

from src.config.config_gemma import Config
from src.document_ingestion.document_processor import DocumentProcessor
from src.vectorstore.vectorstore import VectorStore
from src.graph_builder.graph_builder_adv import GraphBuilder

# Ensure dataset is loaded if not already defined
if 'dataset' not in locals():
    dataset = pd.read_excel('/content/drive/MyDrive/RAG/datasets/datasets.xlsx', sheet_name="final")

# Always re-initialize graph_builder to ensure the latest patches are used
print("Initializing graph_builder...")

# Check/Init LLM and VectorStore
if 'vector_store' not in locals() and 'vector_store' not in globals():
    print("Initializing LLMs and VectorStore...")
    llm = Config.get_llm()
    slm = Config.get_slm()
    embedding = Config.get_embedding_model()

    # Process documents (if not already done/persisted, but we re-run to be safe)
    print("Processing documents...")
    pdfs = '/content/drive/MyDrive/RAG/datasets'
    doc_processor = DocumentProcessor(embedding)
    documents = doc_processor.process(pdfs)

    vector_store = VectorStore(embedding)
    vector_store.create_vectorstore(documents)
else:
    # Ensure we have the references if vector_store already exists
    llm = Config.get_llm()
    slm = Config.get_slm()
    embedding = Config.get_embedding_model()

# Build graph (will use the patched 'build' method)
print("Building graph...")
graph_builder = GraphBuilder(
    retriever=vector_store.get_retriever(),
    llm=llm,
    slm=slm
)
builder = graph_builder.build()
print("Graph built successfully.")

answers = {}


Initializing graph_builder...
Initializing LLMs and VectorStore...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing documents...
Chroma vector store created and persisted to /content/drive/MyDrive/RAG/chroma_db
Building graph...
---BUILDING GRAPH---
---GRAPH COMPILED---
Graph built successfully.
CPU times: user 3min 59s, sys: 16.3 s, total: 4min 15s
Wall time: 7min 43s


<unknown>:19: SyntaxWarning: invalid escape sequence '\s'


In [9]:
# response = graph_builder.run('How does Brazil’s Saeb system contribute to monitoring reading progress across grades, and what subjects and grade levels does it assess according to the 2021 structure?')

In [10]:
%%time

for i, q in enumerate(dataset['question'].tolist()):
    # Skip NaN values in the question list
    if isinstance(q, float) and math.isnan(q):
        print(f"Skipping question at index {i} due to NaN value.")
        continue

    print(f"Processing question {i}: {q}")
    try:
        response = graph_builder.run(q)
        answers[i] = {}
        answers[i]['query'] = response['question']
        answers[i]['answer'] = response['final_answer']
        answers[i]['sub_questions'] = response['sub_questions']
    except Exception as e:
        print(f"Error processing question {i}: {e}")
        answers[i] = {'query': q, 'answer': f"Error: {str(e)}", 'sub_questions': []}


Processing question 0: Which month is set aside each year in the UAE for nationwide reading and literacy activities, including those associated with the Abu Dhabi Reads campaign?
---RETRIEVING DOCUMENTS FOR: Which month is set aside each year in the UAE for nationwide reading and literacy activities, including those associated with the Abu Dhabi Reads campaign?---
--GRADING DOCUMENTS---
---PLANNING SUB-STEPS FOR QUESTION: Which month is set aside each year in the UAE for nationwide reading and literacy activities, including those associated with the Abu Dhabi Reads campaign?---
---GENERATING ANSWERS FOR SUB-QUESTIONS AND FINAL ANSWER---
---GRADING GENERATION---
---GENERATION USEFUL---
Processing question 1: How many public, private, and charter schools exist in Abu Dhabi?
---RETRIEVING DOCUMENTS FOR: How many public, private, and charter schools exist in Abu Dhabi?---
--GRADING DOCUMENTS---
---PLANNING SUB-STEPS FOR QUESTION: How many public, private, and charter schools exist in Abu D

In [11]:
answers = pd.DataFrame(answers).T
answers

,query,answer,sub_questions
0,Which month is set aside each year in the UAE ...,The UAE designates March annually as the month...,{'sub_question_1': {'query': 'What month does ...
1,"How many public, private, and charter schools ...","Based on the provided information, there are a...",{'sub_question_1': {'query': 'How many public ...
2,At which year levels are Australian students a...,,{}
3,What grading scale must Australian schools use...,Australian schools utilize a five-point gradin...,{'sub_question_1': {'query': 'What is the stan...
4,What mandatory early education requirement mus...,Austrian children who fall before March 1st of...,{'sub_question_1': {'query': 'What is the spec...
...,...,...,...
295,"""What is the name of the technology-enabled ap...","Based on the passage, the technology-enabled a...",{'sub_question_1': {'query': 'What does the pa...
296,"""According to the passages, what is the name o...","According to the passages, the law that reauth...",{'sub_question_1': {'query': 'What is the name...
297,"""According to the passages, how did the COVID-...","According to the provided documents, the COVID...",{'sub_question_1': {'query': 'How did the COVI...
298,"According to the document, what age range does...","According to the document, Spain’s basic educa...",{'sub_question_1': {'query': 'What is the gene...


In [12]:
import os
output_path = '/content/drive/MyDrive/RAG/datasets/adv_rag_with_gemma_result_20260101.csv'

answers.to_csv(output_path, index=False)